# Import

In [ ]:
import sys
from pathlib import Path

# Ensure notebooks/week5 is on path so 'utils' package can be imported
_cwd = Path.cwd()
if not (_cwd / "utils").is_dir():
    _week5 = _cwd / "notebooks" / "week5"
    if _week5.exists():
        sys.path.insert(0, str(_week5))

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient
from qdrant_client.models import Prefetch, Filter, FieldCondition, MatchText, FusionQuery, Document
from langsmith import traceable, get_current_run_tree
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.types import Send, Command
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage, convert_to_messages, convert_to_openai_messages
from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List, Optional, Sequence
from IPython.display import Image, display
from operator import add
from openai import OpenAI
import openai
import random
import ast
import inspect
import instructor
import json
from utils.utils import get_tool_descriptions, format_ai_message

# Add to Shopping Cart Tool

In [ ]:
items = [
    {
        "product_id": "B0B96LV4C5",
        "quantity": 2,
    },
    {
        "product_id": "B0BYYLJRHT",
        "quantity": 4
    }
]

In [ ]:
"""
Add to Shopping Cart tool for Week 5 Shopping Cart Agent.

Learning: Agent tools that persist state require a database. This tool:
- Fetches product metadata (price, image) from Qdrant by parent_asin
- Uses tools_database (bootcamp spec) separate from langgraph_db (checkpointer)
- Upsert logic: INSERT new row or UPDATE quantity if (user_id, cart_id, product_id) exists
- Prefetch + FusionQuery: Qdrant hybrid-search pattern for exact product lookup by filter
"""

import numpy as np
import psycopg2
from psycopg2.extras import RealDictCursor
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, Prefetch, FusionQuery


def add_to_shopping_cart(items: list[dict], user_id: str, cart_id: str) -> str:
    """Add a list of provided items to the shopping cart.

    Args:
        items: A list of items to add to the shopping cart. Each item is a dictionary
            with the following keys: product_id, quantity.
        user_id: The id of the user to add the items to the shopping cart.
        cart_id: The id of the shopping cart to add the items to.

    Returns:
        A list of the items added to the shopping cart.
    """
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="langgraph_user",
        password="langgraph_password",
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        for item in items:
            product_id = item["product_id"]
            quantity = item["quantity"]

            qdrant_client = QdrantClient(url="http://localhost:6333")

            # Qdrant lookup: Prefetch with filter by parent_asin (dummy_vector unused; filter does the work)
            # Why Prefetch: hybrid-search collection expects prefetch; filter narrows to single product
            dummy_vector = np.zeros(1536).tolist()
            results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-01-hybrid-search",
                prefetch=[
                    Prefetch(
                        query=dummy_vector,
                        filter=Filter(
                            must=[
                                FieldCondition(
                                    key="parent_asin",
                                    match=MatchValue(value=product_id),
                                )
                            ]
                        ),
                        using="text-embedding-3-small",
                        limit=20,
                    )
                ],
                query=FusionQuery(fusion="rrf"),
                limit=1,
            )
            # Guard: product_id must exist in Qdrant catalog; else IndexError on points[0]
            if not results.points:
                raise ValueError(f"Product {product_id} not found in catalog")
            payload = results.points[0].payload

            product_image_url = payload.get("image")
            price = payload.get("price")
            currency = "USD"

            # Upsert: check if (user_id, cart_id, product_id) exists; UPDATE quantity or INSERT
            check_query = """
            SELECT id, quantity, price
            FROM shopping_carts.shopping_cart_items
            WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
            """
            cursor.execute(check_query, (user_id, cart_id, product_id))
            existing_item = cursor.fetchone()

            if existing_item:
                # Update existing item
                new_quantity = existing_item["quantity"] + quantity
                update_query = """
                UPDATE shopping_carts.shopping_cart_items
                SET
                    quantity = %s,
                    price = %s,
                    currency = %s,
                    product_image_url = COALESCE(%s, product_image_url)
                WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
                RETURNING id, quantity, price
                """
                cursor.execute(
                    update_query,
                    (
                        new_quantity,
                        price,
                        currency,
                        product_image_url,
                        user_id,
                        cart_id,
                        product_id,
                    ),
                )
            else:
                # Insert new item
                insert_query = """
                INSERT INTO shopping_carts.shopping_cart_items (
                    user_id, shopping_cart_id, product_id,
                    price, quantity, currency, product_image_url
                ) VALUES (%s, %s, %s, %s, %s, %s, %s)
                RETURNING id, quantity, price
                """
                cursor.execute(
                    insert_query,
                    (
                        user_id,
                        cart_id,
                        product_id,
                        price,
                        quantity,
                        currency,
                        product_image_url,
                    ),
                )

    conn.close()
    return f"Added {items} to the shopping cart."

In [ ]:
add_to_shopping_cart(items, "abc", "def")

In [ ]:
items_2 = [
    {
        "product_id": "B0B96LV4C5",
        "quantity": 8,
    },
]

In [ ]:
add_to_shopping_cart(items_2, "abc", "def")

In [ ]:
add_to_shopping_cart(items, "abcd", "defg")

# Get the Shopping Card Items Tool

In [ ]:
def get_shopping_cart(user_id: str, cart_id: str) -> list[dict]:
    """Retrieve all items in a user's shopping cart.

    Learning: Read-only tool; returns list of dicts with product_id, price, quantity,
    total_price (price * quantity). RealDictCursor yields dict-like rows for easy serialization.
    Args:
        user_id: User ID
        cart_id: Cart identifier

    Returns:
        List of dictionaries containing cart items
    """
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="langgraph_user",
        password="langgraph_password",
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        query = """
        SELECT
            product_id, price, quantity,
            currency, product_image_url,
            (price * quantity) as total_price
        FROM shopping_carts.shopping_cart_items
        WHERE user_id = %s AND shopping_cart_id = %s
        ORDER BY added_at DESC
        """
        cursor.execute(query, (user_id, cart_id))
        # Convert RealDictRow to plain dict for JSON-serializable return (agent tool output)
        return [dict(row) for row in cursor.fetchall()]

    conn.close()

In [ ]:
get_shopping_cart("abc", "def")

# Deleting Items from the Shopping Cart Tool

In [ ]:
def remove_from_cart(product_id: str, user_id: str, cart_id: str) -> bool:
    """Remove an item completely from the shopping cart.

    Learning: DELETE by (user_id, cart_id, product_id). rowcount > 0 indicates success;
    returns False if item wasn't in cart (idempotent for "remove" semantics).
    Args:
        user_id: User ID
        product_id: Product ID to remove
        cart_id: Cart identifier

    Returns:
        True if item was removed, False if item wasn't found
    """
    conn = psycopg2.connect(
        host="localhost",
        port=5433,
        database="tools_database",
        user="langgraph_user",
        password="langgraph_password",
    )
    conn.autocommit = True

    with conn.cursor(cursor_factory=RealDictCursor) as cursor:
        query = """
        DELETE FROM shopping_carts.shopping_cart_items
        WHERE user_id = %s AND shopping_cart_id = %s AND product_id = %s
        """
        cursor.execute(query, (user_id, cart_id, product_id))
        return cursor.rowcount > 0

    conn.close()

In [ ]:
remove_from_cart("B0B96LV4C5", "abc", "def")